## Fixed Steering Vectors on Gemma

Key fixes:
1. Use `min_token_index` to only steer generation (not the prompt)
2. Train on specific middle layers (not all layers)
3. Use `read_token_index=-1` (last token of prompt)
4. Start with small multipliers (0.1-0.5)


In [ ]:
import torch
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from einops import rearrange, repeat, einsum
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from termcolor import colored
import json

import time
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from tqdm import tqdm

from steering_vectors import SteeringVector, train_steering_vector

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

# Use Gemma 2 (standard architecture) - change this if you want to try Gemma 3
model_name = 'google/gemma-2-2b-it'  # or 'google/gemma-3-4b-it'

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map=device, 
    torch_dtype=torch.bfloat16,
    trust_remote_code=True, 
)

model.eval()  
for param in model.parameters():
    param.requires_grad = False
    
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Check how many layers the model has
num_layers = model.config.num_hidden_layers
print(f"Model has {num_layers} layers")
print(f"Middle layers would be around: {num_layers//2}")

In [ ]:
DATASET = '/home/user/contrastive-pair-gen/data/contrast_pairs/harmfulness_lesswrong.json'

with open(DATASET, 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples")

In [ ]:
# Create pairs - your approach is fine!
pairs = []
for example in data[:20]:  # Use more examples for better steering vector
    harmful_messages = [{"role": "user", "content": example['harmful']}]
    harmless_messages = [{"role": "user", "content": example['harmless']}]
    
    harmful = tokenizer.apply_chat_template(
        harmful_messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    harmless = tokenizer.apply_chat_template(
        harmless_messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    pairs.append((harmful, harmless))

print(f"Created {len(pairs)} pairs")
print(f"\nExample pair:")
print(f"Harmful: {pairs[0][0][:100]}...")
print(f"Harmless: {pairs[0][1][:100]}...")

## Train Steering Vector - WITH FIXES

In [ ]:
# FIX 1: Use specific middle layers (not all layers)
# FIX 2: Use read_token_index=-1 (last token before generation)

middle_layer = num_layers // 2
# Train on a few layers around the middle
layers_to_use = [middle_layer - 1, middle_layer, middle_layer + 1]

print(f"Training on layers: {layers_to_use}")

steering_vector = train_steering_vector(
    model,
    tokenizer,
    pairs,
    layers=layers_to_use,  # FIX 1: Specific layers
    read_token_index=-1,    # FIX 2: Read from last token
    show_progress=True,
)

print(f"\nTrained steering vector on layers: {list(steering_vector.layer_activations.keys())}")

## Test with proper settings

In [ ]:
prompt = [{"role": "user", "content": "Write a poem about hatred."}]
prompt_formatted = tokenizer.apply_chat_template(
    prompt,
    tokenize=False, 
    add_generation_prompt=True
)
inputs = tokenizer(prompt_formatted, return_tensors="pt").to(device)

# FIX 3: We need to know where the prompt ends so we only steer generation
prompt_length = inputs.input_ids.shape[1]
print(f"Prompt length: {prompt_length} tokens")
print(f"Prompt text: {prompt_formatted}")

In [ ]:
# FIX 3: Use min_token_index to only steer generation tokens
# FIX 4: Use smaller multipliers

with torch.no_grad():
    # Baseline
    outputs_baseline = model.generate(
        **inputs, 
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )
    baseline_text = tokenizer.decode(outputs_baseline[0], skip_special_tokens=True)
    
    # Steered AWAY from harmful (negative multiplier) 
    with steering_vector.apply(
        model, 
        multiplier=-0.5,           # FIX 4: Smaller multiplier
        min_token_index=prompt_length  # FIX 3: Only steer generation
    ):
        outputs_harmless = model.generate(
            **inputs, 
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
        harmless_text = tokenizer.decode(outputs_harmless[0], skip_special_tokens=True)
    
    # Steered TOWARD harmful (positive multiplier)
    with steering_vector.apply(
        model, 
        multiplier=0.5,            # FIX 4: Smaller multiplier
        min_token_index=prompt_length  # FIX 3: Only steer generation
    ):
        outputs_harmful = model.generate(
            **inputs, 
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
        harmful_text = tokenizer.decode(outputs_harmful[0], skip_special_tokens=True)

print(colored("=== BASELINE ===", "yellow"))
print(baseline_text)
print(colored("\n=== STEERED HARMLESS (multiplier=-0.5) ===", "green"))
print(harmless_text)
print(colored("\n=== STEERED HARMFUL (multiplier=0.5) ===", "red"))
print(harmful_text)

## Experiment with different multipliers

In [ ]:
# Try a range of multipliers to see the effect
test_multipliers = [-1.0, -0.5, -0.1, 0.0, 0.1, 0.5, 1.0]

print("Testing different multipliers:\n")

for mult in test_multipliers:
    with torch.no_grad():
        if mult == 0.0:
            # No steering
            outputs = model.generate(
                **inputs, 
                max_new_tokens=30,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
        else:
            with steering_vector.apply(model, multiplier=mult, min_token_index=prompt_length):
                outputs = model.generate(
                    **inputs, 
                    max_new_tokens=30,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id
                )
        
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Just show the generated part
        generated = text[len(prompt_formatted):]
        
        color = "red" if mult > 0 else ("green" if mult < 0 else "yellow")
        print(colored(f"Multiplier {mult:+.1f}: {generated[:100]}...", color))
        print()

## Try different layers

Different layers might work better for different tasks.

In [ ]:
# Try early, middle, and late layers
layer_groups = [
    ([num_layers // 4], "early"),
    ([num_layers // 2], "middle"),
    ([3 * num_layers // 4], "late"),
]

for layers, name in layer_groups:
    print(f"\n=== Training on {name} layer(s) {layers} ===")
    
    sv = train_steering_vector(
        model,
        tokenizer,
        pairs[:10],  # Use fewer examples for speed
        layers=layers,
        read_token_index=-1,
        show_progress=False,
    )
    
    with torch.no_grad():
        with sv.apply(model, multiplier=-0.5, min_token_index=prompt_length):
            outputs = model.generate(
                **inputs, 
                max_new_tokens=30,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
            text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            generated = text[len(prompt_formatted):]
            print(colored(f"{name} layers (harmless): {generated[:80]}...", "green"))